In [ ]:
import polars as pl
from datetime import datetime as dt
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
DAILY_DIRECTORY = Path("daily")
DAILY_DIRECTORY.mkdir(exist_ok=True)

In [ ]:
def concat_dfs(directory: Path) -> pl.LazyFrame:
    return pl.scan_parquet(str(directory / "*.parquet")).drop_nulls()

In [ ]:
files = DAILY_DIRECTORY.glob("*.parquet")
df = pl.scan_parquet(files)

In [ ]:
files = DAILY_DIRECTORY.glob("*.parquet")
df = pl.scan_parquet(files)

reuters_bbg = (
    df
    .filter(
        pl.any_horizontal(
            pl.col("url").str.contains("reuters"),
            pl.col("url").str.contains("bloomberg"),
            pl.col("url").str.contains("finance.yahoo")
        )
    )
    .with_columns(
        pl.col("datetime").dt.date().alias("date")
    )
)

daily_counts = (
    reuters_bbg
    .group_by("date")
    .len()
    .sort("date")
)

result = daily_counts.collect(streaming=True, engine="gpu")

In [ ]:
plt.bar(result["date"], result["len"])
plt.title("Reuters/Bloomberg/Yahoo News")
plt.xticks(rotation=45)
plt.xlabel("Date")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
FILTERED_BRAZIL_PATH = Path("brazil_news")
FILTERED_BRAZIL_PATH.mkdir(exist_ok=True)

brazil_news = (
    reuters_bbg
    .filter(
        pl.any_horizontal(
            pl.col("locations").str.contains_any(["brazil"], ascii_case_insensitive=True),
            pl.col("url").str.contains_any(["brazil"], ascii_case_insensitive=True),
            pl.col("themes").str.contains_any(["brazil"], ascii_case_insensitive=True),
            pl.col("v2themes").str.contains_any(["brazil"], ascii_case_insensitive=True),
        )
    )
)

brazil_news.sink_parquet(FILTERED_BRAZIL_PATH / "brazil.parquet", compression_level=22)

In [ ]:
brazil_news = pl.read_parquet(FILTERED_BRAZIL_PATH / "brazil.parquet")

In [ ]:
brazil_counts = (
    brazil_news
    .group_by("date")
    .len()
)

In [ ]:
plt.bar(brazil_counts["date"], brazil_counts["len"])
plt.title("Brazil Reuters/Bloomberg/Yahoo News")
plt.xticks(rotation=45)
plt.xlabel("Date")
plt.ylabel("Count")
plt.tight_layout()
plt.show()